In [ ]:
from pymilvus import MilvusClient
import pymilvus

print("pymilvus 版本:", pymilvus.__version__)

client = MilvusClient(uri="http://192.168.150.101:19530")
print("连接成功，当前 collections:", client.list_collections())

In [ ]:
from pymilvus import MilvusClient, DataType, Function, FunctionType

client = MilvusClient(uri="http://192.168.150.101:19530")
collection_name = "milvus_new"

# 建 Schema：id 自己给，字段必须严格声明
schema = client.create_schema(auto_id=False, enable_dynamic_field=False)

# 5 个字段：主键 / 稠密向量 / 稀疏向量 / 过滤字段 / 原文(中文分词)
schema.add_field(field_name="id",       datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="dense",    datatype=DataType.FLOAT_VECTOR, dim=1024)
schema.add_field(field_name="sparse",   datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="h2",       datatype=DataType.VARCHAR, max_length=128)
schema.add_field(field_name="content",  datatype=DataType.VARCHAR, max_length=2048,
                 enable_analyzer=True, analyzer_params={"type": "chinese"})

# BM25 函数：content 自动转成 sparse，存和查都不用你算
bm25_function = Function(name="text_bm25_emb",
                         input_field_names=["content"],
                         output_field_names=["sparse"],
                         function_type=FunctionType.BM25)
schema.add_function(bm25_function)

print(schema)  # 自查：打印出来看字段和函数齐不齐